In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("Feature engineering") \
  .getOrCreate()

In [ ]:
 data = spark.read.csv("data/housing.csv", header=True, inferSchema=True)

 data.printSchema()

In [ ]:
filtered_data = data.na.drop(subset=['total_bedrooms'])
filtered_data.count()

In [ ]:
train_data, test_data = filtered_data.randomSplit([0.8, 0.2], seed=42)
print("Train size: ", train_data.count())
print("Test size: ", test_data.count())
test_data.show(5)

In [ ]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol='ocean_proximity', outputCol='ocean_proximity_index')
indexer_model = indexer.fit(train_data)
train_data = indexer_model.transform(train_data)
train_data.show()

In [ ]:
train_data.select('ocean_proximity', 'ocean_proximity_index') \
  .distinct() \
  .show()

In [ ]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(inputCol='ocean_proximity_index', outputCol='ocean_proximity_vec', dropLast=False)
encoder_model = encoder.fit(train_data)
train_data = encoder_model.transform(train_data)
train_data.show()

In [ ]:
train_data.select('ocean_proximity', 'ocean_proximity_index', 'ocean_proximity_vec') \
  .distinct() \
  .show()

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_columns = ['housing_median_age', 'total_rooms', 'total_bedrooms',
                   'population', 'households', 'median_income', 'ocean_proximity_vec']
assembler = VectorAssembler(inputCols=feature_columns, outputCol='unscaled_features')

train_data = assembler.transform(train_data)
train_data.show()

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol='unscaled_features', outputCol='features', withMean=True, withStd=True)
scaler_model = scaler.fit(train_data)
transformed_train_data = scaler_model.transform(train_data)
transformed_train_data.show()

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol='features', labelCol='median_house_value')

model = lr.fit(transformed_train_data)

In [ ]:
test_data = indexer_model.transform(test_data)
test_data = encoder_model.transform(test_data)
test_data = assembler.transform(test_data)
test_data = scaler_model.transform(test_data)
test_data.show()

In [ ]:
test_predictions = model.transform(test_data)
test_predictions.show()

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_mae = RegressionEvaluator(labelCol='median_house_value', predictionCol='prediction', metricName='mae')
mae = evaluator_mae.evaluate(test_predictions)
print(f"Mean Absolute Error (MAE): {mae}")